# 06 - List sites and catalogs

Use the sites and catalogs **backends** to list an instance's sites and catalogs
over SCAPI. SCAPI endpoints and the Account Manager token endpoint are mocked
with `respx`. A final cell shows the **synchronous** facade -- the same call with
no `await`.

Public API: `create_catalogs_backend`, `CatalogsBackendConfig`,
`create_sites_backend` / `SitesBackendConfig` (from
`b2c_tooling_sdk.operations.sites`), `to_organization_id`; and the blocking
twins under `b2c_tooling_sdk.sync`.

In [ ]:
# --- Offline, credential-free setup -------------------------------------------
# Everything below runs with NO real network and NO real credentials. HTTP is
# mocked with respx, all state lives in a throwaway temp dir, and the auth-token
# caches are reset -- mirroring the SDK's own test harness (tests/conftest.py).
import base64
import json
import os
import tempfile
import time
from pathlib import Path

import httpx
import respx

from b2c_tooling_sdk.auth.oauth import reset_oauth_cache_for_testing
from b2c_tooling_sdk.auth.oauth_implicit import reset_implicit_cache_for_testing
from b2c_tooling_sdk.auth.oauth_pkce import reset_pkce_cache_for_testing
from b2c_tooling_sdk.auth.session_store import (
    FileAuthSessionBackend,
    set_auth_session_backend,
)

_tmp = Path(tempfile.mkdtemp(prefix="b2c-nb-"))
(_tmp / "data").mkdir(parents=True, exist_ok=True)
(_tmp / "config").mkdir(parents=True, exist_ok=True)

# Point every config/data dir at the temp dir so we never touch a real user store.
os.environ["XDG_DATA_HOME"] = str(_tmp / "data")
os.environ["XDG_CONFIG_HOME"] = str(_tmp / "config")
os.environ["LOCALAPPDATA"] = str(_tmp / "data")
os.environ.pop("B2C_CONFIG_DIR", None)

# Reset the module-level OAuth token caches for deterministic runs.
reset_oauth_cache_for_testing()
reset_pkce_cache_for_testing()
reset_implicit_cache_for_testing()

# Install a temp-dir file-backed auth-session store as the default.
set_auth_session_backend(FileAuthSessionBackend(_tmp / "store"))
print("Isolated temp dir:", _tmp)

In [ ]:
# The SDK decodes (does NOT verify) Account Manager access tokens, so a valid
# JWT *shape* is enough for a mocked token endpoint. This mirrors the unsigned
# test JWTs used by the SDK's own suite (tests/helpers/jwt.py).
def make_jwt(*, expires_in=3600, scope=None, sub=None):
    def b64(raw: bytes) -> str:
        return base64.urlsafe_b64encode(raw).rstrip(b"=").decode("ascii")

    header = {"alg": "RS256", "typ": "JWT"}
    payload = {"exp": int(time.time()) + expires_in}
    if scope is not None:
        payload["scope"] = scope
    if sub is not None:
        payload["sub"] = sub
    return ".".join([b64(json.dumps(header).encode()), b64(json.dumps(payload).encode()), "sig"])

## SCAPI-eligible instance + mocked endpoints

SCAPI needs a `short_code` + `tenant_id` + client-credentials OAuth. The SCAPI
URLs embed the organization id derived from the tenant id.

In [ ]:
from b2c_tooling_sdk import (
    DEFAULT_ACCOUNT_MANAGER_HOST,
    AuthConfig,
    B2CInstance,
    CatalogsBackendConfig,
    InstanceConfig,
    ListCatalogsOptions,
    create_catalogs_backend,
    to_organization_id,
)
from b2c_tooling_sdk.auth import OAuthAuthConfig

# create_sites_backend lives in the sites operations module (not the top-level barrel).
from b2c_tooling_sdk.operations.sites import (
    ListSitesOptions,
    SitesBackendConfig,
    create_sites_backend,
)

SHORT_CODE = "kv7kzm78"
TENANT_ID = "zzxy_prd"
ORG_ID = to_organization_id(TENANT_ID)
print("organization id:", ORG_ID)

instance = B2CInstance(
    InstanceConfig(
        hostname="example.demandware.net",
        short_code=SHORT_CODE,
        tenant_id=TENANT_ID,
        api_backend="scapi",
    ),
    AuthConfig(oauth=OAuthAuthConfig(client_id="cid", client_secret="secret")),
)

TOKEN_URL = f"https://{DEFAULT_ACCOUNT_MANAGER_HOST}/dwsso/oauth2/access_token"
SCAPI = f"https://{SHORT_CODE}.api.commercecloud.salesforce.com"
SITES_URL = f"{SCAPI}/site/sites/v1/organizations/{ORG_ID}/sites"
CATALOGS_URL = f"{SCAPI}/product/catalogs/v1/organizations/{ORG_ID}/catalogs"


def _token(_request: httpx.Request) -> httpx.Response:
    return httpx.Response(
        200,
        json={"access_token": make_jwt(scope="sfcc.sites sfcc.catalogs"), "expires_in": 1800},
    )

## List sites and catalogs (async)

`create_sites_backend` / `create_catalogs_backend` pick the SCAPI backend given
our `preference="scapi"`. The list items are returned rich enough that no
per-item enrichment call is needed.

In [ ]:
sites_payload = {
    "data": [
        {"id": "RefArch", "displayName": {"default": "RefArch"}, "storefrontStatus": "online"},
        {"id": "RefArchGlobal", "displayName": {"default": "RefArch Global"}, "storefrontStatus": "online"},
    ],
    "offset": 0,
    "limit": 50,
    "total": 2,
}
catalogs_payload = {
    "data": [
        {"id": "storefront-catalog-m-en", "name": {"default": "Storefront Catalog"}, "online": True},
        {"id": "master-catalog-m-en", "name": {"default": "Master Catalog"}, "online": True},
    ],
    "offset": 0,
    "limit": 50,
    "total": 2,
}

with respx.mock(assert_all_called=False) as router:
    router.post(TOKEN_URL).mock(side_effect=_token)
    router.get(SITES_URL).mock(return_value=httpx.Response(200, json=sites_payload))
    router.get(CATALOGS_URL).mock(return_value=httpx.Response(200, json=catalogs_payload))

    sites_backend = create_sites_backend(SitesBackendConfig(instance=instance, preference="scapi"))
    catalogs_backend = create_catalogs_backend(CatalogsBackendConfig(instance=instance, preference="scapi"))

    sites = await sites_backend.list_sites(ListSitesOptions(count=50))
    catalogs = await catalogs_backend.list_catalogs(ListCatalogsOptions(count=50))

print("Sites:")
for s in sites:
    print(f"  - {s.id} ({s.display_name}) [{s.storefront_status}]")
print("Catalogs:")
for c in catalogs:
    print(f"  - {c.id} ({c.name}) online={c.online}")

assert [s.id for s in sites] == ["RefArch", "RefArchGlobal"]
assert [c.id for c in catalogs] == ["storefront-catalog-m-en", "master-catalog-m-en"]

## The synchronous facade (no `await`)

`b2c_tooling_sdk.sync` mirrors the public API as blocking twins that run on a
background event loop. Factories like `create_catalogs_backend` return a backend
whose methods block instead of returning coroutines -- handy from scripts or a
REPL. (Token caching / single-flight semantics are preserved.)

In [ ]:
from b2c_tooling_sdk.sync import create_catalogs_backend as sync_create_catalogs_backend

with respx.mock(assert_all_called=False) as router:
    router.post(TOKEN_URL).mock(side_effect=_token)
    router.get(CATALOGS_URL).mock(return_value=httpx.Response(200, json=catalogs_payload))

    sync_backend = sync_create_catalogs_backend(CatalogsBackendConfig(instance=instance, preference="scapi"))
    # NOTE: no `await` -- this call blocks and returns the list directly.
    sync_catalogs = sync_backend.list_catalogs(ListCatalogsOptions(count=50))

print("Catalogs (sync):", [c.id for c in sync_catalogs])
assert [c.id for c in sync_catalogs] == ["storefront-catalog-m-en", "master-catalog-m-en"]

## Recap

- `create_sites_backend` / `create_catalogs_backend` selected the SCAPI backend
  and listed sites and catalogs.
- The `b2c_tooling_sdk.sync` facade offers the same factory as a blocking twin --
  identical arguments, no `await`.